In [1]:
import json
import gzip
import pandas as pd


In [2]:
def to_int(text):
    if text is None:
        return None
    num = ''
    started = False
    has_dot = False

    for ch in text:
        if ch.isdigit():
            num += ch
            started = True

        elif ch in '.,' and started and not has_dot:
            num += '.'
            has_dot = True

        elif started:
            break

    if not num:
        return None
    return float(num)

In [3]:
def value_to_text(value):
    
    if value is None:
        return None

    if isinstance(value, list):
        parts = []

        for item in value:
            if item is None:
                continue
            
            text = value_to_text(item)
            if text:
                parts.append(text)

        return "; ".join(parts)

    if isinstance(value, dict):
        parts = []
        for key, val in value.items():
            if not key:
                continue

            if val is None:
                continue

            text = value_to_text(val)
            if text:
                parts.append(f"{key}: {text}")

        return "; ".join(parts)
    return str(value)

In [4]:
def extract_fields(line, fields):

    line = json.loads(line)
    wline = {}

    for field in fields:
        value = line.get(field)
        if field == 'price':
            try: 
                float(value)
            except:
                value = to_int(value)

        elif field == 'details':
            value = value_to_text(value)
            
        wline[field] = value

    return wline

In [5]:
def load(path,l_path, fields,i):
    with gzip.open(path,'rt') as file:
        k=0
        buf = []
        n=1
        for line in file:
            wline = extract_fields(line, fields)
            buf.append(wline)
            k+=1
            if k >= 1_000_000:
                df = pd.DataFrame(buf)
                df.to_parquet(f'{l_path}/part_{n}.parquet', index=False)
                n+=1
                k=0
                buf = []
        if len(buf) != 0:        
            df = pd.DataFrame(buf)
            df.to_parquet(f'{l_path}/part_{n}.parquet',index=False)



In [6]:
path = ['/home/user/mle/amazon2023/meta_All_Beauty.jsonl.gz', '/home/user/mle/amazon2023/All_Beauty.jsonl.gz',
        '/home/user/mle/amazon2023/meta_Books.jsonl.gz', '/home/user/mle/amazon2023/Books.jsonl.gz', 
        '/home/user/mle/amazon2023/meta_Electronics.jsonl.gz', '/home/user/mle/amazon2023/Electronics.jsonl.gz']


path_to_l = ['/home/user/mle/data/All_Beauty/meta', '/home/user/mle/data/All_Beauty/otz', 
             '/home/user/mle/data/Books/meta','/home/user/mle/data/Books/otz',
             '/home/user/mle/data/Electronic/meta','/home/user/mle/data/Electronic/otz' ]

fields = [['main_category','average_rating','parent_asin', 'title', 'details', 'price', 'categories'],
          ['rating', 'verified_purchase', 'user_id', 'parent_asin', 'timestamp']]

In [59]:
for i in range(len(path)):
    load(path[i], path_to_l[i], fields[i%2], i)

In [66]:
data = pd.read_parquet('/home/user/mle/data/Books/meta')

In [ ]:
data